In [112]:
# Load env variables and create client
import os
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv(override=True)

client = Anthropic()
model = os.getenv("MODEL", "claude-haiku-4-5")

model

'claude-fable-5'

In [101]:
# Helper functions
from anthropic.types import Message

def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message
    }
    messages.append(user_message)

def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message
    }
    messages.append(assistant_message)


def chat(messages, system=None, stop_sequences=[], tools=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "stop_sequences": stop_sequences,
    }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message

def text_from_message(message):
    return "\n".join(
        [block.text for block in message.content if block.type == "text"]
    )

In [102]:
# Tools and Schemas

from datetime import datetime, timedelta


def add_duration_to_datetime(
    datetime_str, duration=0, unit="days", input_format="%Y-%m-%d"
):
    date = datetime.strptime(datetime_str, input_format)

    if unit == "seconds":
        new_date = date + timedelta(seconds=duration)
    elif unit == "minutes":
        new_date = date + timedelta(minutes=duration)
    elif unit == "hours":
        new_date = date + timedelta(hours=duration)
    elif unit == "days":
        new_date = date + timedelta(days=duration)
    elif unit == "weeks":
        new_date = date + timedelta(weeks=duration)
    elif unit == "months":
        month = date.month + duration
        year = date.year + month // 12
        month = month % 12
        if month == 0:
            month = 12
            year -= 1
        day = min(
            date.day,
            [
                31,
                29 if year % 4 == 0 and (year % 100 != 0 or year % 400 == 0) else 28,
                31,
                30,
                31,
                30,
                31,
                31,
                30,
                31,
                30,
                31,
            ][month - 1],
        )
        new_date = date.replace(year=year, month=month, day=day)
    elif unit == "years":
        new_date = date.replace(year=date.year + duration)
    else:
        raise ValueError(f"Unsupported time unit: {unit}")

    return new_date.strftime("%A, %B %d, %Y %I:%M:%S %p")


def set_reminder(content, timestamp):
    print(f"----\nSetting the following reminder for {timestamp}:\n{content}\n----")


add_duration_to_datetime_schema = {
    "name": "add_duration_to_datetime",
    "description": "Adds a specified duration to a datetime string and returns the resulting datetime in a detailed format. This tool converts an input datetime string to a Python datetime object, adds the specified duration in the requested unit, and returns a formatted string of the resulting datetime. It handles various time units including seconds, minutes, hours, days, weeks, months, and years, with special handling for month and year calculations to account for varying month lengths and leap years. The output is always returned in a detailed format that includes the day of the week, month name, day, year, and time with AM/PM indicator (e.g., 'Thursday, April 03, 2025 10:30:00 AM').",
    "input_schema": {
        "type": "object",
        "properties": {
            "datetime_str": {
                "type": "string",
                "description": "The input datetime string to which the duration will be added. This should be formatted according to the input_format parameter.",
            },
            "duration": {
                "type": "number",
                "description": "The amount of time to add to the datetime. Can be positive (for future dates) or negative (for past dates). Defaults to 0.",
            },
            "unit": {
                "type": "string",
                "description": "The unit of time for the duration. Must be one of: 'seconds', 'minutes', 'hours', 'days', 'weeks', 'months', or 'years'. Defaults to 'days'.",
            },
            "input_format": {
                "type": "string",
                "description": "The format string for parsing the input datetime_str, using Python's strptime format codes. For example, '%Y-%m-%d' for ISO format dates like '2025-04-03'. Defaults to '%Y-%m-%d'.",
            },
        },
        "required": ["datetime_str"],
    },
}

set_reminder_schema = {
    "name": "set_reminder",
    "description": "Creates a timed reminder that will notify the user at the specified time with the provided content. This tool schedules a notification to be delivered to the user at the exact timestamp provided. It should be used when a user wants to be reminded about something specific at a future point in time. The reminder system will store the content and timestamp, then trigger a notification through the user's preferred notification channels (mobile alerts, email, etc.) when the specified time arrives. Reminders are persisted even if the application is closed or the device is restarted. Users can rely on this function for important time-sensitive notifications such as meetings, tasks, medication schedules, or any other time-bound activities.",
    "input_schema": {
        "type": "object",
        "properties": {
            "content": {
                "type": "string",
                "description": "The message text that will be displayed in the reminder notification. This should contain the specific information the user wants to be reminded about, such as 'Take medication', 'Join video call with team', or 'Pay utility bills'.",
            },
            "timestamp": {
                "type": "string",
                "description": "The exact date and time when the reminder should be triggered, formatted as an ISO 8601 timestamp (YYYY-MM-DDTHH:MM:SS) or a Unix timestamp. The system handles all timezone processing internally, ensuring reminders are triggered at the correct time regardless of where the user is located. Users can simply specify the desired time without worrying about timezone configurations.",
            },
        },
        "required": ["content", "timestamp"],
    },
}

batch_tool_schema = {
    "name": "batch_tool",
    "description": "Invoke multiple other tool calls simultaneously",
    "input_schema": {
        "type": "object",
        "properties": {
            "invocations": {
                "type": "array",
                "description": "The tool calls to invoke",
                "items": {
                    "type": "object",
                    "properties": {
                        "name": {
                            "type": "string",
                            "description": "The name of the tool to invoke",
                        },
                        "arguments": {
                            "type": "string",
                            "description": "The arguments to the tool, encoded as a JSON string",
                        },
                    },
                    "required": ["name", "arguments"],
                },
            }
        },
        "required": ["invocations"],
    },
}

pass

In [103]:
# get_current_datetime tool and schema
from anthropic.types import ToolParam

def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("Date format cannot be empty")
    return datetime.now().strftime(date_format)

get_current_datetime_schema = ToolParam({
    "name": "get_current_datetime",
    "description": "Get the current date and time, formatted as a string. Call this when the user asks what the current date, time, day of the week, or timestamp is, or when a task needs the current moment as a reference point (e.g. calculating a duration from 'now').",
    "input_schema": {
      "type": "object",
      "properties": {
        "date_format": {
          "type": "string",
          "description": "A Python strftime format string controlling how the returned datetime is rendered, e.g. '%Y-%m-%d %H:%M:%S' for '2025-04-03 14:30:00', or '%B %d, %Y' for 'April 03, 2025'. Must not be empty. Defaults to '%Y-%m-%d %H:%M:%S' if omitted."
        }
      },
      "required": []
    }
  }
)



In [104]:
messages = []

add_user_message(messages, "Hello, Claude! Can you tell me what the current date and time is?")
response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema],
)

add_assistant_message(messages, response.content)

messages

[{'role': 'user',
  'content': 'Hello, Claude! Can you tell me what the current date and time is?'},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_01UUze1TH93CZVTqxC3fPQsW', caller=DirectCaller(type='direct'), input={}, name='get_current_datetime', type='tool_use')]}]

In [105]:
import json

def run_tool(tool_name, tool_input):
    if tool_name == "get_current_datetime":
        return get_current_datetime(**tool_input)
    if tool_name == "add_duration_to_datetime":
        return add_duration_to_datetime(**tool_input)
    if tool_name == "set_reminder":
        return set_reminder(**tool_input)

def run_tools(message):
    tool_requests = [
        block for block in message.content if block.type == "tool_use"
    ]
    tool_result_blocks = []

    for tool_request in tool_requests:
        try:
            #tool_output = get_current_datetime(**tool_request.input)
            tool_output = run_tool(tool_request.name, tool_request.input)
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": json.dumps(tool_output),
                "is_error": False,
            }
        except Exception as e:
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": f"Error: {e}",
                "is_error": True,
            }

        tool_result_blocks.append(tool_result_block)

    return tool_result_blocks

In [106]:
def run_conversation(messages):
    tools = [get_current_datetime_schema, add_duration_to_datetime_schema, set_reminder_schema]
    while True:
        response = chat(messages, tools=tools)

        add_assistant_message(messages, response)
        print(text_from_message(response))

        if response.stop_reason != "tool_use":
            break

        tool_results = run_tools(response)
        add_user_message(messages, tool_results)
    return messages

In [118]:
messages = []
add_user_message(
    messages,
    "my birthday is February 5. what day will this fall on for the next 5 years?"
    # "What is the current time in HH:MM format? Also, what is the current time in SS format?"
)
run_conversation(messages)




Here's what day February 5 falls on for your next 5 birthdays:

- **2027** – Friday 🎉
- **2028** – Saturday
- **2029** – Monday
- **2030** – Tuesday
- **2031** – Wednesday

Nice — you've got a Friday and a Saturday birthday coming up in the next two years! (Note the jump from Saturday to Monday between 2028 and 2029 — that's because 2028 is a leap year, so the day of the week skips ahead by two instead of one.)


[{'role': 'user',
  'content': 'my birthday is February 5. what day will this fall on for the next 5 years?'},
 {'role': 'assistant',
  'content': [ThinkingBlock(signature='CAIS4gQKjwEIERgCKkAhcnZid2VYFlndIjSer01cfbDc9WlbbEk3cDD+cScgt1ZX4VdKdZdP+QwcXWHM+3vXivW0HoZkR8BtCTiOQVLoMg5jbGF1ZGUtZmFibGUtNTgBQgh0aGlua2luZ1okNjc0MDQyMjYtMjhiMC00NzVkLWI2ZDEtMjU4MGU3NDIxZTY5qAHwkpHVBhIMlldYn5HxNnFAWLCfGgyx8EqS6XrG63EJdmAiMD4lNxQje6pwkSAIZICuJ1EE5znFr2Y8BNKxV4BVMeCjAlOiJDjOcwHioNrTM4XKiSr/Ah6aSDyiqTQIvGamMvkbvDGOQU8jhLuHrn234bjfsy6j5cNe+i9wPKNk5tKo6prr7TLIZH3jNwYkzuWNjZzO5o+v4bTOQKc/cbFIL9UtR+4svn8VVz/ggFr9nGLDK3A9stRlUD1M0g711YinlgIsI/KLm0fK9MtBp4o9iDirNp+IAz/UYqA8WgasVYCNUvdoIGPg6EePm+XkA+buJ/AjKmyiInNl1874dg2p5S178NfJ7wDdbjc35N+wkLuHL0JrKMqatkHX2GOCRJGhFKtoeHy8kitg+GZtptzlZGxqQxrdUN8WjBuAGWu5jn8+C61x+iE866rLAzWTdqSk8R7/FjKDsIQbEEpFiDse//pqdWPwh5oDzgDIV/qcYGnFLg76UXmXyyjiRBoKdQ//SrwGjcJz5S8Cwn1BSYyj5WdFwLFvfsJyBMyN1urHem4xqf/OuBfKD7klCBw8D4B2gy7Chm7Imav1LQHUekKzPnP1UXNpNLZwKYoZiqwZ16OYUL9DGAE=', 